# Explore raw SMPP / SS7 CDRs

Goal: look at real row shapes before finalizing `cdr_mapping.py`'s cleaning rules.

Already confirmed for **SMPP**: `smpp_operation` 4/5 are the request rows that actually
carry `decoded_content` + the rule engine's `decision` — these are what passes through
the network and is worth revenue-losing-spam analysis. `80000004`/`80000005` are the
ack/response rows (only carry `message_id`, no content) — not signal, drop them.

**Still open for SS7**: `message_type` doesn't split as cleanly — `decision` is populated
on almost every type, but `decoded_content` only shows up on types 2 and 3. This notebook
is for eyeballing sample rows per type to figure out what 1/2/3/4/5/7 actually mean before
we decide what to keep.

In [2]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

RAW_DIR = Path("..") / "data" / "raw"

## SMPP

In [3]:
smpp_path = RAW_DIR / "SMPP" / "0208" / "stg_smpp_20260802_0500.csv"
smpp = pd.read_csv(smpp_path, low_memory=False)
smpp.head(3)

,index,time_stamp,smpp_operation,result,source_ip,source_port,dest_ip,dest_port,system_id,instance_id,virtual_gt,sequence_no,message_id,oa_ton,oa_npi,oa,da_ton,da_npi,da,dcs,sar_ref,sar_msg_parts,sar_msg_part,app_dest_port,app_src_port,decision,content,decoded_content,rule,fraud_type,esme_class,msg_type,gsm_features,messaging_mode,inverted,create_date,message_state,receipted_message_id,rule_name
0,0899814883409718_20260802050003_master2,2026-08-02 05:00:03.281,4,0,175.50.50.3,57802,172.28.17.236,2775,Esms12,8689845505807057_20260802033206_master1,2962497612196134_20251209000902_master1,630,NaN,2.0,0.0,66688,1.0,1.0,6.014907e+10,-15.0,NaN,NaN,NaN,NaN,NaN,0.0,050003390201524d302041434f4d2053444e204248443a204869204d5548414d4d4144204146...,"é@¥9$£RM0 ACOM SDN BHD: Hi MUHAMMAD AFFIQ AIMAN BIN HAMDAN, your repayment D...",SW_0410120618449599_20251209005906_master1,NaN,64.0,0.0,1.0,0.0,0.0,2026-08-02 05:00:21.030891,NaN,NaN,Esms12 A2P Domestic
1,0173590685419419_20260802050003_master2,2026-08-02 05:00:03.296,80000004,0,175.50.50.3,57802,172.28.17.236,2775,Esms12,8689845505807057_20260802033206_master1,2962497612196134_20251209000902_master1,630,8cd002a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-02 05:00:21.030891,NaN,NaN,NaN
2,0261642653745967_20260802050003_master2,2026-08-02 05:00:03.759,80000005,0,62.67.222.27,18147,172.28.17.236,2775,gtsmax,9870807004563150_20260731030905_master1,0385037707206803_20251202020252_master1,15226343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-02 05:00:21.030891,NaN,NaN,NaN


In [6]:
smpp['dcs'].value_counts()

dcs
-15.0    16300
 0.0      1189
 25.0      266
 8.0       236
-11.0       34
Name: count, dtype: int64

In [4]:
print(smpp["smpp_operation"].value_counts(dropna=False))
print()
smpp.groupby("smpp_operation").agg(
    has_content=("decoded_content", lambda s: s.notna().sum()),
    has_message_id=("message_id", lambda s: s.notna().sum()),
    has_decision=("decision", lambda s: s.notna().sum()),
    n=("index", "count"),
)

smpp_operation
4           15601
80000004    15601
80000005     2424
5            2424
Name: count, dtype: int64



,has_content,has_message_id,has_decision,n
smpp_operation,,,,
4,15601,0,15601,15601
5,2418,0,2424,2424
80000004,0,15586,0,15601
80000005,0,0,0,2424


In [5]:
# one sample row per operation type, transposed so all columns are visible
for op, group in smpp.groupby("smpp_operation"):
    print(f"=== smpp_operation = {op} ===")
    display(group.iloc[[0]].T)

=== smpp_operation = 4 ===


,0
index,0899814883409718_20260802050003_master2
time_stamp,2026-08-02 05:00:03.281
smpp_operation,4
result,0
source_ip,175.50.50.3
source_port,57802
dest_ip,172.28.17.236
dest_port,2775
system_id,Esms12
instance_id,8689845505807057_20260802033206_master1


=== smpp_operation = 5 ===


,3
index,0841433558854639_20260802050004_master2
time_stamp,2026-08-02 05:00:04.019
smpp_operation,5
result,0
source_ip,175.50.50.3
source_port,52382
dest_ip,172.28.17.236
dest_port,2775
system_id,Esms6
instance_id,5179351458150734_20260802040447_master1


=== smpp_operation = 80000004 ===


,1
index,0173590685419419_20260802050003_master2
time_stamp,2026-08-02 05:00:03.296
smpp_operation,80000004
result,0
source_ip,175.50.50.3
source_port,57802
dest_ip,172.28.17.236
dest_port,2775
system_id,Esms12
instance_id,8689845505807057_20260802033206_master1


=== smpp_operation = 80000005 ===


,2
index,0261642653745967_20260802050003_master2
time_stamp,2026-08-02 05:00:03.759
smpp_operation,80000005
result,0
source_ip,62.67.222.27
source_port,18147
dest_ip,172.28.17.236
dest_port,2775
system_id,gtsmax
instance_id,9870807004563150_20260731030905_master1


In [6]:
content_rows = smpp[smpp["decoded_content"].notna()]
print("decision value counts (content-bearing rows only):")
print(content_rows["decision"].value_counts(dropna=False))
print()
print("rule_name value counts (top 15):")
print(content_rows["rule_name"].value_counts(dropna=False).head(15))
print()
print("fraud_type value counts:")
print(content_rows["fraud_type"].value_counts(dropna=False))

decision value counts (content-bearing rows only):
decision
0.0    18005
1.0       14
Name: count, dtype: int64

rule_name value counts (top 15):
rule_name
Esms3 A2P Domestic          2906
NaN                         2657
Esms9 A2P Domestic          2300
Esms5 A2P Domestic          1724
Esms6 A2P Domestic          1606
Esms8 A2P Domestic          1567
Esms10 A2P Domestic         1060
Esms12 A2P Domestic          891
Esms A2P Domestic            818
Esms13a A2P Domestic         741
Esms4 A2P Domestic           687
Esms7 A2P Domestic           366
short_code_5digit_GTSmax     294
Esms11 A2P Domestic          285
Esms2 A2P Domestic           103
Name: count, dtype: int64

fraud_type value counts:
fraud_type
NaN     18005
spam       14
Name: count, dtype: int64


## SS7

In [7]:
ss7_path = RAW_DIR / "SS7" / "0208" / "stg_ss7_20260802_0000.csv"
ss7 = pd.read_csv(ss7_path, low_memory=False, nrows=200_000)
print(ss7.shape)
ss7.head(3)

(200000, 35)


,index,time_stamp,message_type,reference,calling_gt,smsc,msisdn,b_number,dcs,sarref,msg_part,msg_parts,content,raw_user_data,decision,rule,fraud_type,status,create_date,update_date,file_name,decoded_content,called_gt,imsi,virtual_imsi,vlr_address,virtual_vlr_outbound_smsc_gt,ton,npi,pid,tpdu_length,error1,error2,delivery_code,action_code
0,0064539208996038_20260802000148_master1,2026-08-02 00:01:48,3,3083064413,60120000665,6.012000e+10,60126750760,0126107438,0,NaN,0,0,d3f2b82e4fd3f3a0797e4e2fb741f437485e6ea7dd6450fe5dd73514d474bbac93c164b6170c...,d3f2b82e4fd3f3a0797e4e2fb741f437485e6ea7dd6450fe5dd73514d474bbac93c164b6170c...,0,NaN,NaN,NaN,2026-08-02 00:01:51.308986,NaN,NaN,Security system to remind you:\r\nTime:2026/08/02 00:02:22\r\nDevName:SEN303...,60120000015,5.021214e+14,NaN,NaN,NaN,0.0,1,0,117,NaN,NaN,0,0
1,0122236234029221_20260802000149_master1,2026-08-02 00:01:48,5,3089538884,60160782883,6.016078e+10,NaN,601133711926,0,NaN,0,0,NaN,NaN,0,NaN,NaN,NaN,2026-08-02 00:01:51.308986,NaN,NaN,NaN,60120000020,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,0,0
2,0853752554235408_20260802000149_master1,2026-08-02 00:01:48,3,3073327812,60120000665,6.012000e+10,60123726942,0195000153,0,NaN,0,0,c4e23408b226a7d420088a04318bd66223485d0d9bc7694f048be56a30182c569b891ac4e234...,c4e23408b226a7d420088a04318bd66223485d0d9bc7694f048be56a30182c569b891ac4e234...,1,S_6206658820146276_20251209044058_master1,spam,NaN,2026-08-02 00:01:51.308986,NaN,NaN,"DESA VISTA PH LEVE\rAT+CMGS=""0195000153""\rDESA VISTA PH LEVEL HH OK",60120000015,5.021216e+14,NaN,NaN,NaN,0.0,1,0,65,NaN,NaN,0,5


In [8]:
print(ss7["message_type"].value_counts(dropna=False))
print()
ss7.groupby("message_type").agg(
    has_content=("decoded_content", lambda s: s.notna().sum()),
    has_decision=("decision", lambda s: s.notna().sum()),
    n=("index", "count"),
)

message_type
1    74866
5    74853
3    36138
2     8137
4     6004
7        2
Name: count, dtype: int64



,has_content,has_decision,n
message_type,,,
1,0,74866,74866
2,8135,8137,8137
3,36135,36138,36138
4,0,6004,6004
5,0,74853,74853
7,0,2,2


In [9]:
# one sample row per message_type, transposed — look at calling_gt/called_gt/
# b_number/smsc/status/action_code/error1/error2/delivery_code/imsi/vlr_address
# to work out what each type actually represents (content-bearing message vs.
# routing query/ack/delivery report).
for mt, group in ss7.groupby("message_type"):
    print(f"=== message_type = {mt}  (n={len(group)}) ===")
    display(group.iloc[[0]].T)

=== message_type = 1  (n=74866) ===


,3
index,0710306301809879_20260802000149_master1
time_stamp,2026-08-02 00:01:48
message_type,1
reference,3091068031
calling_gt,60120000065
smsc,NaN
msisdn,NaN
b_number,NaN
dcs,0
sarref,NaN


=== message_type = 2  (n=8137) ===


,9
index,0362199856756632_20260802000149_master1
time_stamp,2026-08-02 00:01:49
message_type,2
reference,3088280243
calling_gt,60192030553
smsc,60192030553.0
msisdn,60145690885
b_number,60179631395
dcs,0
sarref,NaN


=== message_type = 3  (n=36138) ===


...

,22
index,0598381534665117_20260802000149_master1
time_stamp,2026-08-02 00:01:49
message_type,4
reference,3088280243
calling_gt,60120000062
smsc,NaN
msisdn,NaN
b_number,NaN
dcs,0
sarref,NaN


=== message_type = 5  (n=74853) ===


,1
index,0122236234029221_20260802000149_master1
time_stamp,2026-08-02 00:01:48
message_type,5
reference,3089538884
calling_gt,60160782883
smsc,60160782883.0
msisdn,NaN
b_number,601133711926
dcs,0
sarref,NaN


=== message_type = 7  (n=2) ===


,108114
index,0413398867796938_20260802003002_master2
time_stamp,2026-08-02 00:30:02
message_type,7
reference,2514211804
calling_gt,60120001128
smsc,NaN
msisdn,NaN
b_number,NaN
dcs,0
sarref,NaN


In [10]:
# status/decision/action_code patterns per type — another angle on the same question
for col in ["status", "action_code", "decision"]:
    if col in ss7.columns:
        print(f"--- {col} by message_type ---")
        print(ss7.groupby("message_type")[col].value_counts(dropna=False))
        print()

--- status by message_type ---
message_type  status
1             NaN       74866
2             NaN        8137
3             NaN       36138
4             NaN        6004
5             NaN       74853
7             NaN           2
Name: count, dtype: int64

--- action_code by message_type ---
message_type  action_code
1              0             65486
               1              5536
               6              2601
               27              657
               13              585
               34                1
2              0              6462
               5               720
              -4               490
               34              465
3              0             30839
               5              5279
               34               12
              -5                 8
4              0              3181
               6              1298
               27             1211
               32              189
               31               90
              

## Next

Once it's clear which SS7 `message_type` codes are the real message-bearing / revenue-
relevant events, fold that into `cdr_mapping.py` the same way the SMPP 4/5-only filter
will be — as an explicit cleaning step (`clean_ss7_raw`) run before `map_ss7_to_canonical`,
not baked silently into the mapping function itself.

## SAR populated for Message Type - 4

In [7]:
op4 = smpp[smpp["smpp_operation"] == 4]

print("op4 rows:", len(op4))
print("sar_ref populated:", op4["sar_ref"].notna().sum())
print("sar_msg_parts populated:", op4["sar_msg_parts"].notna().sum())
print("sar_msg_part populated:", op4["sar_msg_part"].notna().sum())
print("gsm_features == 1 (UDH-based concat):", (op4["gsm_features"] == 1).sum())

op4 rows: 15601
sar_ref populated: 0
sar_msg_parts populated: 0
sar_msg_part populated: 0
gsm_features == 1 (UDH-based concat): 2405


In [8]:
from pathlib import Path

results = []
for f in sorted((RAW_DIR / "SMPP").rglob("*.csv")):
    df = pd.read_csv(
        f, low_memory=False,
        usecols=["smpp_operation", "sar_ref", "sar_msg_parts", "sar_msg_part", "gsm_features"],
    )
    op4 = df[df["smpp_operation"] == 4]
    results.append({
        "file": f.name,
        "op4_rows": len(op4),
        "sar_ref_populated": op4["sar_ref"].notna().sum(),
        "gsm_features_eq_1": (op4["gsm_features"] == 1).sum(),
    })

summary = pd.DataFrame(results)
print(summary)
print()
print("totals:", summary[["op4_rows", "sar_ref_populated", "gsm_features_eq_1"]].sum())

                          file  op4_rows  sar_ref_populated  gsm_features_eq_1
0   stg_smpp_20260802_0000.csv     55563                  0               6942
1   stg_smpp_20260802_0100.csv     36359                  0               3952
2   stg_smpp_20260802_0200.csv     29620                  0               4189
3   stg_smpp_20260802_0300.csv     23550                  0               3332
4   stg_smpp_20260802_0400.csv     18405                  0               3212
5   stg_smpp_20260802_0500.csv     15601                  0               2405
6   stg_smpp_20260802_0600.csv     24174                  0               5489
7   stg_smpp_20260802_0700.csv     58096                  3               5083
8   stg_smpp_20260802_0800.csv    144850                  0              11750
9   stg_smpp_20260802_0900.csv    177891                  0              28690
10  stg_smpp_20260802_1000.csv    196699                  0              30259
11  stg_smpp_20260802_1100.csv    164688            

In [14]:
nonzero = summary[summary["sar_ref_populated"] > 0]
print(nonzero)

                          file  op4_rows  sar_ref_populated  gsm_features_eq_1
7   stg_smpp_20260802_0700.csv     58096                  3               5083
11  stg_smpp_20260802_1100.csv    164688                  4              19384
13  stg_smpp_20260802_1300.csv    160750                  4              23768
17  stg_smpp_20260802_1700.csv    144036                  2              13546
19  stg_smpp_20260802_1900.csv    148332                  4              16482
21  stg_smpp_20260802_2100.csv    106618                  2               9636
42  stg_smpp_20260803_1800.csv    181041                  6              31959
44  stg_smpp_20260803_2000.csv    170393                  6              20656


In [18]:
f = RAW_DIR / "SMPP" / "0208" / nonzero.iloc[0]["file"]   # adjust path prefix if file is under a dated subfolder
df = pd.read_csv(f, low_memory=False)
op4 = df[df["smpp_operation"] == 4]
sar_rows = op4[op4["sar_ref"].notna()]

In [21]:
sar_rows[["sar_ref", "sar_msg_parts", "sar_msg_part",  "oa", "da"]]

,sar_ref,sar_msg_parts,sar_msg_part,oa,da
72180,529.0,1.0,2.0,67425,6.017875e+10
72236,529.0,1.0,2.0,67425,6.017875e+10
72239,529.0,1.0,2.0,67425,6.017875e+10


In [24]:
import pandas as pd
from pathlib import Path

results = []
for f in sorted((RAW_DIR / "SMPP").rglob("*.csv")):
    df = pd.read_csv(f, low_memory=False, usecols=["smpp_operation", "fraud_type"])
    op4 = df[df["smpp_operation"] == 4]
    counts = op4["fraud_type"].value_counts(dropna=False)
    results.append({"file": f.name, **counts.to_dict()})

summary = pd.DataFrame(results).fillna(0)
print(summary)
print()
print("totals across all files:")
print(summary.drop(columns="file").sum())


# all_fraud_types = []
# for f in sorted((RAW_DIR / "SMPP").rglob("*.csv"))
#     df = pd.read_csv(f, low_memory=False, usecols=["smpp_operation", "fraud_type"])
#     op4 = df[df["smpp_operation"] == 4]
#     all_fraud_types.append(op4["fraud_type"])

# pd.concat(all_fraud_types).value_counts(dropna=False) 

                          file     NaN  spam
0   stg_smpp_20260802_0000.csv   55485    78
1   stg_smpp_20260802_0100.csv   36309    50
2   stg_smpp_20260802_0200.csv   29562    58
3   stg_smpp_20260802_0300.csv   23530    20
4   stg_smpp_20260802_0400.csv   18395    10
5   stg_smpp_20260802_0500.csv   15587    14
6   stg_smpp_20260802_0600.csv   24158    16
7   stg_smpp_20260802_0700.csv   58072    24
8   stg_smpp_20260802_0800.csv  144816    34
9   stg_smpp_20260802_0900.csv  177809    82
10  stg_smpp_20260802_1000.csv  196619    80
11  stg_smpp_20260802_1100.csv  164586   102
12  stg_smpp_20260802_1200.csv  160452   108
13  stg_smpp_20260802_1300.csv  160644   106
14  stg_smpp_20260802_1400.csv  155193   100
15  stg_smpp_20260802_1500.csv  142243    60
16  stg_smpp_20260802_1600.csv  135888    54
17  stg_smpp_20260802_1700.csv  143970    66
18  stg_smpp_20260802_1800.csv  145177    66
19  stg_smpp_20260802_1900.csv  148284    48
20  stg_smpp_20260802_2000.csv  138275    72
21  stg_sm